# Experimento 1: Problema XOR

## Motor de Redes Neuronales - Optimización y Heurística
**Autores:** Raúl Mendoza, Adrián Ojeda, Varela  
**Universidad de Las Palmas de Gran Canaria**

---

## 1. Introducción

El problema XOR (OR exclusivo) es un problema clásico de clasificación binaria no linealmente separable. Es importante históricamente porque demostró las limitaciones del perceptrón simple y motivó el desarrollo de redes multicapa.

**Tabla de verdad XOR:**
| x₁ | x₂ | XOR(x₁, x₂) |
|----|----|-------------|
| 0  | 0  | 0           |
| 0  | 1  | 1           |
| 1  | 0  | 1           |
| 1  | 1  | 0           |

Este experimento valida que nuestra implementación de forward pass y backpropagation funciona correctamente.

## 2. Importación de Módulos

Importamos las clases de nuestro motor de redes neuronales implementado en `src/`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.network import NeuralNetwork
from src.layers import Dense
from src.optimizers import Adam
from src.trainer import Trainer

np.random.seed(42)

print("Módulos importados correctamente ✓")

## 3. Definición del Dataset XOR

El dataset XOR consta de solo 4 muestras. Usamos codificación one-hot para las etiquetas ya que trataremos el problema como clasificación con 2 clases.

In [ ]:
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1],
], dtype=np.float32)

y = np.array([
    [1, 0],
    [0, 1],
    [0, 1],
    [1, 0],
], dtype=np.float32)

print("Dataset XOR:")
print("="*40)
for i in range(len(X)):
    label = np.argmax(y[i])
    print(f"  Entrada: {X[i]} → Clase: {label}")

## 4. Definición de la Arquitectura de Red

Para resolver XOR necesitamos al menos una capa oculta, ya que el problema no es linealmente separable.

**Arquitectura seleccionada:**
- **Capa de entrada:** 2 neuronas (x₁, x₂)
- **Capa oculta:** 4 neuronas con activación tanh
- **Capa de salida:** 2 neuronas con activación softmax (para clasificación binaria)

In [ ]:
net = NeuralNetwork()

net.add(Dense(n_in=2, n_out=4, activation="tanh", weight_init="xavier"))

net.add(Dense(n_in=4, n_out=2, activation="softmax", weight_init="xavier"))

print("Arquitectura de la red:")
print("="*40)
print("  Entrada:  2 neuronas")
print("  Oculta:   4 neuronas (tanh)")
print("  Salida:   2 neuronas (softmax)")
print(f"\nTotal parámetros: {sum(p.size for p in net.params())}")

## 5. Configuración del Entrenamiento

Utilizamos el optimizador Adam con una tasa de aprendizaje de 0.05 y la función de pérdida Cross-Entropy.

In [ ]:
optimizer = Adam(lr=0.05)
trainer = Trainer(net, optimizer, loss_name="cross_entropy")

print("Configuración del entrenamiento:")
print("="*40)
print(f"  Optimizador: Adam (lr=0.05)")
print(f"  Pérdida: Cross-Entropy")
print(f"  Épocas: 500")
print(f"  Batch size: 4 (dataset completo)")

## 6. Entrenamiento de la Red

Entrenamos la red durante 500 épocas. Como el dataset tiene solo 4 muestras, usamos batch_size=4 (batch gradient descent).

In [ ]:
print("Iniciando entrenamiento...\n")
train_losses, _ = trainer.train(
    X, y,
    epochs=500,
    batch_size=4,
    verbose=False
)

print(f"\nEntrenamiento completado!")
print(f"  Pérdida inicial: {train_losses[0]:.4f}")
print(f"  Pérdida final:   {train_losses[-1]:.6f}")

## 7. Evaluación y Resultados

Verificamos que la red clasifica correctamente las 4 entradas del problema XOR.

In [ ]:
predictions = net.forward(X)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y, axis=1)

print("Resultados de la clasificación:")
print("="*60)
print(f"{'Entrada':<12} {'Prob. Clase 0':<15} {'Prob. Clase 1':<15} {'Pred':<6} {'Real':<6} {'✓/✗'}")
print("-"*60)

correct = 0
for i in range(len(X)):
    is_correct = predicted_classes[i] == true_classes[i]
    correct += is_correct
    symbol = "✓" if is_correct else "✗"
    print(f"{str(X[i]):<12} {predictions[i,0]:<15.4f} {predictions[i,1]:<15.4f} {predicted_classes[i]:<6} {true_classes[i]:<6} {symbol}")

print("-"*60)
accuracy = correct / len(X) * 100
print(f"\nPrecisión: {correct}/{len(X)} = {accuracy:.1f}%")

## 8. Visualización de la Curva de Pérdida

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, 'b-', linewidth=2)
plt.xlabel('Época', fontsize=12)
plt.ylabel('Pérdida (Cross-Entropy)', fontsize=12)
plt.title('Curva de Pérdida - Problema XOR', fontsize=14)
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.savefig('../memoria/fig_loss_xor.png', dpi=150)
plt.close()

print("Gráfica guardada en: memoria/fig_loss_xor.png")

## 9. Visualización de la Frontera de Decisión

In [ ]:
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 100), np.linspace(-0.5, 1.5, 100))
grid_points = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)

Z = net.forward(grid_points)
Z = np.argmax(Z, axis=1).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.4, cmap='RdBu')
plt.contour(xx, yy, Z, colors='k', linewidths=0.5)

colors = ['red' if c == 0 else 'blue' for c in true_classes]
plt.scatter(X[:, 0], X[:, 1], c=colors, s=200, edgecolors='black', linewidths=2, zorder=5)

for i in range(len(X)):
    plt.annotate(f'XOR={true_classes[i]}', (X[i,0]+0.05, X[i,1]+0.1), fontsize=10)

plt.xlabel('x₁', fontsize=12)
plt.ylabel('x₂', fontsize=12)
plt.title('Frontera de Decisión Aprendida - XOR', fontsize=14)
plt.xlim(-0.5, 1.5)
plt.ylim(-0.5, 1.5)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../memoria/fig_decision_xor.png', dpi=150)
plt.close()

print("Gráfica guardada en: memoria/fig_decision_xor.png")

## 10. Conclusiones del Experimento XOR

### Resultados obtenidos:
- La red neuronal converge correctamente, reduciendo la pérdida a valores cercanos a cero.
- Se clasifican correctamente las 4 muestras del problema XOR (100% de precisión).
- La frontera de decisión muestra la capacidad de la red para aprender funciones no lineales.

### Validación de la implementación:
Este experimento demuestra que:
1. ✅ El **forward pass** propaga correctamente las activaciones.
2. ✅ El **backpropagation** calcula correctamente los gradientes.
3. ✅ El **optimizador Adam** actualiza los parámetros de forma efectiva.
4. ✅ Las **funciones de activación** (tanh, softmax) funcionan correctamente.
5. ✅ La **función de pérdida** Cross-Entropy se calcula y deriva correctamente.